## There is a factory website that has several machines each running the same number of processes. Write a solution to find the average time each machine takes to complete a process.

The time to complete a process is the 'end' timestamp minus the 'start' timestamp. The average time is calculated by the total time to complete every process on the machine divided by the number of processes that were run.

The resulting table should have the machine_id along with the average time as processing_time, which should be rounded to 3 decimal places.

Return the result table in any order.

The result format is in the following example.

 

Example 1:

Input: 
Activity table:
+------------+------------+---------------+-----------+
| machine_id | process_id | activity_type | timestamp |
+------------+------------+---------------+-----------+
| 0          | 0          | start         | 0.712     |
| 0          | 0          | end           | 1.520     |
| 0          | 1          | start         | 3.140     |
| 0          | 1          | end           | 4.120     |
| 1          | 0          | start         | 0.550     |
| 1          | 0          | end           | 1.550     |
| 1          | 1          | start         | 0.430     |
| 1          | 1          | end           | 1.420     |
| 2          | 0          | start         | 4.100     |
| 2          | 0          | end           | 4.512     |
| 2          | 1          | start         | 2.500     |
| 2          | 1          | end           | 5.000     |
+------------+------------+---------------+-----------+
Output: 
+------------+-----------------+
| machine_id | processing_time |
+------------+-----------------+
| 0          | 0.894           |
| 1          | 0.995           |
| 2          | 1.456           |
+------------+-----------------+

## taking pands dataframe from leetcode and then converting to spark dataframe 

In [14]:
import pandas as pd
data = [[0, 0, 'start', 0.712], [0, 0, 'end', 1.52], [0, 1, 'start', 3.14], [0, 1, 'end', 4.12], [1, 0, 'start', 0.55], [1, 0, 'end', 1.55], [1, 1, 'start', 0.43], [1, 1, 'end', 1.42], [2, 0, 'start', 4.1], [2, 0, 'end', 4.512], [2, 1, 'start', 2.5], [2, 1, 'end', 5]]
activity = pd.DataFrame(data, columns=['machine_id', 'process_id', 'activity_type', 'timestamp']).astype({'machine_id':'Int64', 'process_id':'Int64', 'activity_type':'object', 'timestamp':'Float64'})

In [6]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("AVG Time processing").getOrCreate()
activity_df=spark.createDataFrame(activity)
activity_df.show()


+----------+----------+-------------+---------+
|machine_id|process_id|activity_type|timestamp|
+----------+----------+-------------+---------+
|         0|         0|        start|    0.712|
|         0|         0|          end|     1.52|
|         0|         1|        start|     3.14|
|         0|         1|          end|     4.12|
|         1|         0|        start|     0.55|
|         1|         0|          end|     1.55|
|         1|         1|        start|     0.43|
|         1|         1|          end|     1.42|
|         2|         0|        start|      4.1|
|         2|         0|          end|    4.512|
|         2|         1|        start|      2.5|
|         2|         1|          end|      5.0|
+----------+----------+-------------+---------+



In [7]:
from pyspark.sql.functions import col
df_start=activity_df.withColumn("start_time",col("timestamp")).filter(col("activity_type")=='start')
df_end=activity_df.withColumn("end_time",col("timestamp")).filter(col("activity_type")=='end')
df_start.show()
df_end.show()

+----------+----------+-------------+---------+----------+
|machine_id|process_id|activity_type|timestamp|start_time|
+----------+----------+-------------+---------+----------+
|         0|         0|        start|    0.712|     0.712|
|         0|         1|        start|     3.14|      3.14|
|         1|         0|        start|     0.55|      0.55|
|         1|         1|        start|     0.43|      0.43|
|         2|         0|        start|      4.1|       4.1|
|         2|         1|        start|      2.5|       2.5|
+----------+----------+-------------+---------+----------+

+----------+----------+-------------+---------+--------+
|machine_id|process_id|activity_type|timestamp|end_time|
+----------+----------+-------------+---------+--------+
|         0|         0|          end|     1.52|    1.52|
|         0|         1|          end|     4.12|    4.12|
|         1|         0|          end|     1.55|    1.55|
|         1|         1|          end|     1.42|    1.42|
|         

In [17]:
from pyspark.sql.functions import sum
#from pyspark.sql.window import Window
#window_spec=Window.orderBy("machine_id")
df_start_sum=df_start.groupBy("machine_id").agg(sum("start_time").alias("total_start_time"))
df_end_sum=df_end.groupBy("machine_id").agg(sum("end_time").alias("total_end_time"))
df_start_sum.show()
df_end_sum.show()


+----------+------------------+
|machine_id|  total_start_time|
+----------+------------------+
|         0|3.8520000000000003|
|         1|              0.98|
|         2|               6.6|
+----------+------------------+

+----------+------------------+
|machine_id|    total_end_time|
+----------+------------------+
|         0| 5.640000000000001|
|         1|2.9699999999999998|
|         2|             9.512|
+----------+------------------+



In [20]:
df3=df_start_sum.join(df_end_sum,"machine_id","inner")\
.withColumn("processing_time",col("total_end_time")-col("total_start_time")/2)
df3.show()

+----------+------------------+------------------+------------------+
|machine_id|  total_start_time|    total_end_time|   processing_time|
+----------+------------------+------------------+------------------+
|         0|3.8520000000000003| 5.640000000000001|3.7140000000000004|
|         1|              0.98|2.9699999999999998|2.4799999999999995|
|         2|               6.6|             9.512| 6.212000000000001|
+----------+------------------+------------------+------------------+

